# Data prep

## Extract all Tenk10k variants

In [37]:
import os
import re
import pickle
import subprocess
import pandas as pd
from glob import glob
import numpy as np
from tqdm import tqdm
from scipy.stats import mannwhitneyu


def generate_cmd(snp_data, model_fn, genome, output_prefix):
    cmd = f"""
source /illumina/scratch/deep_learning/lchen4/illumina_dl_tools/jupyterlab/deeplearningfinal_cuda11.bashrc
eval "$(micromamba shell hook --shell bash)"
micromamba activate tf_2_8
SNP_DATA={snp_data}
MODEL_H5={model_fn}
GENOME={genome}
OUTPUT_PREFIX={output_prefix}
OUTPUT_PICKLE={output_prefix}_predictions_at_snp.pkl
OUTPUT_TSV={output_prefix}_snp_scores.tsv
echo chrombpnet snp_score -snps $SNP_DATA -m $MODEL_H5 -g $GENOME -op $OUTPUT_PREFIX
chrombpnet snp_score -snps $SNP_DATA -m $MODEL_H5 -g $GENOME -op $OUTPUT_PREFIX

# keep the log count predictions
python /illumina/scratch/deep_learning/lchen4/tenk10k/scpipe/chrombpnet_simplify_pickle.py $OUTPUT_PICKLE

# taking too much space
gzip $OUTPUT_TSV
rm $OUTPUT_PICKLE 
"""
    return cmd

In [4]:
cell_type_counts_fn = "data/combined_obs_cell_type_counts.tsv"
cell_type_counts = pd.read_csv(cell_type_counts_fn, sep='\t')
cell_type_counts['label'] = cell_type_counts['predicted.id'] + ' (N=' + cell_type_counts['count'].astype(str) + ')'
cell_type_counts = cell_type_counts.set_index("predicted.id")

In [5]:
cell_type_counts

,count,label
predicted.id,,
CD14_Mono,767874,CD14_Mono (N=767874)
CD4_TCM,543523,CD4_TCM (N=543523)
CD4_Naive,457715,CD4_Naive (N=457715)
NK,432705,NK (N=432705)
CD8_TEM,357483,CD8_TEM (N=357483)
CD16_Mono,180128,CD16_Mono (N=180128)
B_naive,152441,B_naive (N=152441)
B_intermediate,114963,B_intermediate (N=114963)
CD4_CTL,105062,CD4_CTL (N=105062)


In [3]:
indirs = [
          "/illumina/scratch/deep_learning/lchen4/tenk10k/data/wgs/december2024_freeze_common_variants/",
          "/illumina/scratch/deep_learning/lchen4/tenk10k/data/wgs/december2024_freeze_rare_variants/"
         ]
fns = []
for d in indirs:
    fns.extend(glob(f"{d}/chr*.vcf.bgz"))

In [22]:
job_dir = "/illumina/scratch/deep_learning/lchen4/tenk10k/scripts/chrombpnet_scoring/extract_variants/"
if not os.path.exists(job_dir): os.makedirs(job_dir, exist_ok=True)
out_dir = "/illumina/scratch/deep_learning/lchen4/tenk10k/data/derived/december2024_freeze_variants/"

for i, fn in enumerate(fns):
    if not os.path.exists(out_dir): os.makedirs(out_dir, exist_ok=True)
    out_tsv = os.path.join(out_dir, os.path.basename(fn).replace(".vcf.bgz", ".tsv.gz"))
    with open(os.path.join(job_dir, f"variant_tsv_{i+1}.sh"), 'w') as fh:
        cmd = r"bcftools query -f '%CHROM\t%POS\t%REF\t%ALT\n'"+ f" {fn} | gzip > {out_tsv}"
        fh.write(cmd)

In [23]:
print(i+1)

48


In [25]:
# !qsub -t 1-48 /illumina/scratch/deep_learning/lchen4/tenk10k/scripts/extract_variants.sh

Your job-array 54187.2-48:1 ("extract_tenk10k_variants") has been submitted


## Convert variants to chromBPNet input tsv files


In [10]:
indirs = [
          "/illumina/scratch/deep_learning/lchen4/tenk10k/data/derived/december2024_freeze_variants/",
         ]
fns = []
for d in indirs:
    fns.extend(glob(f"{d}/*.gz"))

In [ ]:
genome = "/illumina/scratch/deep_learning/lchen4/data/hg38.fa"
out_script_dir = "/illumina/scratch/deep_learning/lchen4/tenk10k/scripts/chrombpnet_scoring/december2024_freeze_variants/"
os.makedirs(out_script_dir, exist_ok=True)
outdir = "/illumina/scratch/deep_learning/lchen4/tenk10k/result/ATAC_ChromBPNet_variant_prediction/december2024_freeze_variants/"
variant_outdir = os.path.join(outdir, "variants")
score_outdir = os.path.join(outdir, "scores")
os.makedirs(variant_outdir, exist_ok=True)
cell_types = cell_type_counts.index
folds = range(0, 5)
cell_type_without_models = []

i = 1
fns = sorted(fns)
for cell_type in cell_types:
    for fn in fns:
        fn_prefix = os.path.basename(fn).split(".tsv.gz")[0]
        snp_data = os.path.join(variant_outdir, fn_prefix+'_snps.tsv.gz')

        if not os.path.exists(snp_data):
            df = pd.read_csv(fn, sep='\t', compression='gzip', header=None)
            df['is_snp'] = df.apply(lambda x: (len(x[2]) == 1) & (len(x[3]) == 1), axis=1)
            df = df[df['is_snp']]
            chrom = df[0].astype(str).values[0]
            if not chrom.startswith('chr'):
                df[0] = 'chr' + chrom
                
            if chrom in ['chrX', 'chrY']:
                print(f'Stripping chr from {chrom}')
                df[0] = df[0].str.lstrip('chr')
    
            dft = df[[0, 1, 2, 3]]
            #dft = dft.drop_duplicates()
            dft[1] = dft[1] - 1   # 1-based to 0-based
            print(f"{fn_prefix}: {len(dft)}")
            print(f"Writing to {snp_data}")
            
            dft.to_csv(snp_data, sep='\t', index=False, header=False)

        for fold in folds:
            celltype_outdir = os.path.join(score_outdir, cell_type, f'fold{fold}')
            os.makedirs(celltype_outdir, exist_ok=True)
            model_fn = f'/illumina/scratch/deep_learning/lchen4/tenk10k/result/ATAC_ChromBPNet/{cell_type}/'\
                       f'chrombpnet_model_full_bias_NKFull1_fold{fold}/models/chrombpnet_nobias.h5'
            if os.path.exists(model_fn):
                output_prefix = os.path.join(celltype_outdir, fn_prefix)
                expected_out_fn = f"{output_prefix}_snp_scores.tsv.gz"
                if not os.path.exists(expected_out_fn):
                    cmd = generate_cmd(snp_data, model_fn, genome, output_prefix)
                    out_script_fn = os.path.join(out_script_dir, f"chrombpnet_scoring_{i}.sh")
                    with open(out_script_fn, 'w') as fh:
                        fh.write(cmd)
                    i+=1
                else:
                    #print(f"{cell_type}, fold {fold} output exists: {expected_out_fn}")
                    pass
            else:
                if cell_type not in cell_type_without_models:
                    cell_type_without_models.append(cell_type)
                    print(f"{cell_type}'s fold {fold} model file is not found")             

## Run scoring

### submit variant scoring jobs

In [ ]:
genome = "/illumina/scratch/deep_learning/lchen4/data/hg38.fa"
out_script_dir = "/illumina/scratch/deep_learning/lchen4/tenk10k/scripts/chrombpnet_scoring/december2024_freeze_variants_patch_6/"
os.makedirs(out_script_dir, exist_ok=True)
outdir = "/illumina/scratch/deep_learning/lchen4/tenk10k/result/ATAC_ChromBPNet_variant_prediction/december2024_freeze_variants/"
variant_outdir = os.path.join(outdir, "variants")
score_outdir = os.path.join(outdir, "scores")
os.makedirs(variant_outdir, exist_ok=True)
cell_types = cell_type_counts.index
folds = range(0, 5)
cell_type_without_models = []

i = 1
fns = sorted(fns)
for cell_type in cell_types:
    for fn in fns:
        fn_prefix = os.path.basename(fn).split(".tsv.gz")[0]
        snp_data = os.path.join(variant_outdir, fn_prefix+'_snps.tsv.gz')
        if ('chrX' not in snp_data) and ('chrY' not in snp_data):
            assert os.path.exists(snp_data)
    
            for fold in folds:
                celltype_outdir = os.path.join(score_outdir, cell_type, f'fold{fold}')
                os.makedirs(celltype_outdir, exist_ok=True)
                model_fn = f'/illumina/scratch/deep_learning/lchen4/tenk10k/result/ATAC_ChromBPNet/{cell_type}/'\
                           f'chrombpnet_model_full_bias_NKFull1_fold{fold}/models/chrombpnet_nobias.h5'
                
                if os.path.exists(model_fn):
                    output_prefix = os.path.join(celltype_outdir, fn_prefix)
                    expected_out_fn = f"{output_prefix}_snp_scores.tsv.gz"
                    if not os.path.exists(expected_out_fn):
                        cmd = generate_cmd(snp_data, model_fn, genome, output_prefix)
                        out_script_fn = os.path.join(out_script_dir, f"chrombpnet_scoring_{i}.sh")
                        with open(out_script_fn, 'w') as fh:
                            fh.write(cmd)
                        i+=1
print(f"Total to run {i-1}")

In [14]:
# !qsub -t 1-2 -tc 2 /illumina/scratch/deep_learning/lchen4/tenk10k/scripts/run_chrombpnet_all_tenk10k_snp_scoring_patch_qsub.sh

Your job-array 221079.1-2:1 ("chrombpnet_tenk10k_allsnps") has been submitted


In [ ]:
# !qsub -t 3-522 /illumina/scratch/deep_learning/lchen4/tenk10k/scripts/run_chrombpnet_all_tenk10k_snp_scoring_patch_qsub.sh

## Average scores across folds

In [ ]:
indirs = [
          "/illumina/scratch/deep_learning/lchen4/tenk10k/data/derived/december2024_freeze_variants/",
         ]
fns = []
for d in indirs:
    fns.extend(glob(f"{d}/*.gz"))
fns = sorted(fns)

genome = "/illumina/scratch/deep_learning/lchen4/data/hg38.fa"
out_script_dir = "/illumina/scratch/deep_learning/lchen4/tenk10k/scripts/chrombpnet_scoring/december2024_freeze_variants/"
os.makedirs(out_script_dir, exist_ok=True)
outdir = "/illumina/scratch/deep_learning/lchen4/tenk10k/result/ATAC_ChromBPNet_variant_prediction/december2024_freeze_variants/"
variant_outdir = os.path.join(outdir, "variants")
score_outdir = os.path.join(outdir, "scores")
os.makedirs(variant_outdir, exist_ok=True)
cell_types = cell_type_counts.index
folds = range(0, 5)

records = []


i = 1
for cell_type in cell_types:
    for fn in fns:
        fn_prefix = fn_prefix = os.path.basename(fn).split(".tsv.gz")[0]
        snp_data = os.path.join(variant_outdir, fn_prefix+'_snps.tsv.gz')
        for fold in folds:
            celltype_outdir = os.path.join(score_outdir, cell_type, f'fold{fold}')
            os.makedirs(celltype_outdir, exist_ok=True)
            model_fn = f'/illumina/scratch/deep_learning/lchen4/tenk10k/result/ATAC_ChromBPNet/{cell_type}/'\
                       f'chrombpnet_model_full_bias_NKFull1_fold{fold}/models/chrombpnet_nobias.h5'
            if os.path.exists(model_fn):
                output_prefix = os.path.join(celltype_outdir, fn_prefix)
                expected_out_fn = f"{output_prefix}_snp_scores.tsv.gz"
                freq = os.path.basename(expected_out_fn).split("_")[1]
                
                if os.path.exists(expected_out_fn):
                    records.append({'celltype': cell_type,
                                    'fold': fold,
                                    'freq': freq,
                                    'expected_fn':expected_out_fn,
                                    'model': model_fn,
                                    'status': 'Done'})
                else:
                    records.append({'celltype': cell_type,
                                    'fold': fold,
                                    'freq': freq,
                                    'expected_fn':expected_out_fn,
                                    'model': model_fn,
                                    'status': 'Missing'})
            else:
                records.append({'celltype': cell_type,
                                'fold': fold,
                                'freq': freq,
                                'expected_fn':expected_out_fn,
                                'model': model_fn,
                                'status': 'No model'})
df_info = pd.DataFrame(records)
df_info['chrom'] = df_info.apply(lambda x: os.path.basename(x['expected_fn']).split("_")[0], axis=1)

In [33]:
df_info['status'].value_counts()

status
Done        6240
No model     480
Name: count, dtype: int64

In [32]:
df_info.head(2)

,celltype,fold,freq,expected_fn,model,status,chrom
0,CD14_Mono,0,common,/illumina/scratch/deep_learning/lchen4/tenk10k...,/illumina/scratch/deep_learning/lchen4/tenk10k...,Done,chr10
1,CD14_Mono,1,common,/illumina/scratch/deep_learning/lchen4/tenk10k...,/illumina/scratch/deep_learning/lchen4/tenk10k...,Done,chr10


In [38]:
# common variants
dft_all = []
outfn = '/illumina/scratch/deep_learning/lchen4/tenk10k/result/ATAC_ChromBPNet_variant_prediction/december2024_freeze_variants/scores/common_snp_scores_all_contexts.parquet'
for ct, grp_ct in df_info[(df_info['freq'] == 'common') & (df_info['status'] == 'Done')].groupby('celltype'):
    t_outdir = os.path.dirname(os.path.dirname(grp_ct['expected_fn'].values[0]))
    t_outfn = os.path.join(t_outdir, f'{ct}_common_variants_snp_scores.parquet')
    print(f"Writing to {t_outfn}")
    dft = []
    for chrom, grp in tqdm(grp_ct.groupby('chrom')):
        if len(grp) == 5: # 5 folds are there
            ts = []
            for ind, row in grp.iterrows():
                fn = row['expected_fn']
                t = pd.read_csv(fn, sep='\t')
                ts.append(t)
            t = pd.concat(ts)
            t = t.groupby(['CHR', 'POS0', 'REF', 'ALT'])\
                 .agg({'log_counts_diff': 'mean', 
                      'log_probs_diff_abs_sum': 'mean', 
                       'probs_jsd_diff': 'mean'}).reset_index()
            t['celltype'] = ct
            dft.append(t)
    dft = pd.concat(dft)
    dft_all.append(dft)
    dft.to_parquet(t_outfn)
dft_all = pd.concat(dft_all)
dft_all.to_parquet(outfn)

Writing to /illumina/scratch/deep_learning/lchen4/tenk10k/result/ATAC_ChromBPNet_variant_prediction/december2024_freeze_variants/scores/ASDC/ASDC_common_variants_snp_scores.parquet


100%|██████████| 24/24 [00:56<00:00,  2.37s/it]


Writing to /illumina/scratch/deep_learning/lchen4/tenk10k/result/ATAC_ChromBPNet_variant_prediction/december2024_freeze_variants/scores/B_intermediate/B_intermediate_common_variants_snp_scores.parquet


100%|██████████| 24/24 [01:09<00:00,  2.89s/it]


Writing to /illumina/scratch/deep_learning/lchen4/tenk10k/result/ATAC_ChromBPNet_variant_prediction/december2024_freeze_variants/scores/B_memory/B_memory_common_variants_snp_scores.parquet


...

100%|██████████| 24/24 [01:08<00:00,  2.87s/it]


Writing to /illumina/scratch/deep_learning/lchen4/tenk10k/result/ATAC_ChromBPNet_variant_prediction/december2024_freeze_variants/scores/gdT/gdT_common_variants_snp_scores.parquet


100%|██████████| 24/24 [01:09<00:00,  2.91s/it]


Writing to /illumina/scratch/deep_learning/lchen4/tenk10k/result/ATAC_ChromBPNet_variant_prediction/december2024_freeze_variants/scores/pDC/pDC_common_variants_snp_scores.parquet


100%|██████████| 24/24 [01:08<00:00,  2.84s/it]


In [40]:
cts = dft_all['celltype'].unique()
cts

array(['ASDC', 'B_intermediate', 'B_memory', 'B_naive', 'CD14_Mono',
       'CD16_Mono', 'CD4_CTL', 'CD4_Naive', 'CD4_Proliferating',
       'CD4_TCM', 'CD4_TEM', 'CD8_Naive', 'CD8_TCM', 'CD8_TEM', 'HSPC',
       'MAIT', 'NK', 'NK_CD56bright', 'NK_Proliferating', 'Plasmablast',
       'Treg', 'cDC1', 'cDC2', 'dnT', 'gdT', 'pDC'], dtype=object)

In [41]:
cts.shape

(26,)

In [42]:
set(cell_type_counts.index) - set(cts)

{'CD8_Proliferating', 'ILC'}

In [ ]:
# rare variants
dft_all = []
outfn = '/illumina/scratch/deep_learning/lchen4/tenk10k/result/ATAC_ChromBPNet_variant_prediction/december2024_freeze_variants/scores/rare_snp_scores_all_contexts.parquet'
for ct, grp_ct in df_info[(df_info['freq'] == 'rare') & (df_info['status'] == 'Done')].groupby('celltype'):
    t_outdir = os.path.dirname(os.path.dirname(grp_ct['expected_fn'].values[0]))
    t_outfn = os.path.join(t_outdir, f'{ct}_rare_variants_snp_scores.parquet')
    dft = []
    for chrom, grp in grp_ct.groupby('chrom'):
        print(chrom, grp.shape)
        if len(grp) == 5: # 5 folds are there
            ts = []
            for ind, row in grp.iterrows():
                fn = row['expected_fn']
                t = pd.read_csv(fn, sep='\t')
                ts.append(t)
            t = pd.concat(ts)
            t = t.groupby(['CHR', 'POS0', 'REF', 'ALT'])\
                 .agg({'log_counts_diff': 'mean', 
                      'log_probs_diff_abs_sum': 'mean', 
                       'probs_jsd_diff': 'mean'}).reset_index()
            t['celltype'] = ct
            dft.append(t)
    dft = pd.concat(dft)
    dft_all.append(dft)
    dft.to_parquet(t_outfn)
dft_all = pd.concat(dft_all)
dft_all.to_parquet(outfn)

In [48]:
dft_all.shape

(1123914168, 8)